# Check 2 — User text and Physio summary correspondance validation



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Setup — install and import libraries

In [ ]:
# Cell 0: Install and import dependencies
!pip install -q transformers datasets accelerate scikit-learn pandas numpy torch

import re
import json
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
from sklearn.metrics import classification_report, confusion_matrix
from scipy.special import softmax

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

PyTorch Version: 2.11.0+cu128
CUDA Available: True


**What this cell does:** imports everything we'll need across both methods, and fixes random seeds so your results are reproducible run to run. `CUDA available` tells you if fine-tuning will use GPU (much faster) or CPU (slower but fine for distilgpt2 at this data size).

# **Using Class-weighted loss function and finetuned model with classification head**

In [ ]:
# Cell 1: Load datasets, deduplicate, and assign unique external user texts
import pandas as pd
import numpy as np

# Ensure your RANDOM_STATE is defined if running this cell standalone
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

REALISTIC_PATH = "/content/drive/MyDrive/user_text_physio_summaries.csv"
FAR_PATH = "/content/drive/MyDrive/far_variant_data_clean.csv"
CLOSE_PATH = "/content/drive/MyDrive/close_variant_impossible_point.csv"
EXTERNAL_TEXTS_PATH = "/content/drive/MyDrive/only_user_texts_frm_Extes.csv" # New file

# 1. Load Realistic Data
pos_df = pd.read_csv(REALISTIC_PATH)[["user_text", "physio_summary"]].dropna()
pos_df = pos_df.drop_duplicates(subset=["user_text", "physio_summary"]).reset_index(drop=True)
pos_df["category"] = "Realistic"
pos_df["label"] = 1

# 2. Load External User Texts (Drop NA and exact duplicates)
ext_texts_df = pd.read_csv(EXTERNAL_TEXTS_PATH)[["user_text"]].dropna()
ext_texts_df = ext_texts_df.drop_duplicates(subset=["user_text"]).reset_index(drop=True)
available_texts = ext_texts_df["user_text"].tolist()

# 3. Load Negative Summaries
far_df = pd.read_csv(FAR_PATH)[["physio_summary"]].dropna()
far_df = far_df.drop_duplicates(subset=["physio_summary"]).reset_index(drop=True)
far_df["category"] = "Far Variant"
far_df["label"] = 0

close_df = pd.read_csv(CLOSE_PATH)[["physio_summary"]].dropna()
close_df = close_df.drop_duplicates(subset=["physio_summary"]).reset_index(drop=True)
close_df["category"] = "Close Variant"
close_df["label"] = 0

# 4. Sample and Assign Unique External Texts
total_needed = len(far_df) + len(close_df)

# Sanity Check: Ensure we have enough unique external texts
if total_needed > len(available_texts):
    raise ValueError(f"Not enough unique external texts! We need {total_needed}, but only have {len(available_texts)}.")

# Sample without replacement so no text is ever reused
sampled_texts = np.random.choice(available_texts, size=total_needed, replace=False)

# Split the sampled texts between the far and close dataframes
far_df["user_text"] = sampled_texts[:len(far_df)]
close_df["user_text"] = sampled_texts[len(far_df):]

# Reorder columns to match pos_df
far_df = far_df[["user_text", "physio_summary", "category", "label"]]
close_df = close_df[["user_text", "physio_summary", "category", "label"]]

print(f"Loaded Clean Summaries & Assigned External Texts:")
print(f"- Realistic Pairs: {len(pos_df)}")
print(f"- Far Variants (Mismatched): {len(far_df)}")
print(f"- Close Variants (Mismatched): {len(close_df)}")
print(f"- Remaining unused external texts: {len(available_texts) - total_needed}")

Loaded Clean Summaries & Assigned External Texts:
- Realistic Pairs: 946
- Far Variants (Mismatched): 1500
- Close Variants (Mismatched): 998
- Remaining unused external texts: 81805


In [ ]:
# Cell 2: Test split and multi-pair negative sampling (NO INFLATION)
from sklearn.model_selection import train_test_split

# 1. Held-out Test Set (100 Realistic, 50 Far, 50 Close)
pos_test = pos_df.sample(n=100, random_state=RANDOM_STATE)
pos_rem = pos_df.drop(pos_test.index).reset_index(drop=True)

# Assign random user_text strictly from the test set to avoid leakage
far_test = far_df.sample(n=50, random_state=RANDOM_STATE)
far_rem = far_df.drop(far_test.index).reset_index(drop=True)
far_test["user_text"] = np.random.choice(pos_test["user_text"].tolist(), size=len(far_test), replace=True)

close_test = close_df.sample(n=50, random_state=RANDOM_STATE)
close_rem = close_df.drop(close_test.index).reset_index(drop=True)
close_test["user_text"] = np.random.choice(pos_test["user_text"].tolist(), size=len(close_test), replace=True)

merged_test_df = pd.concat([pos_test, far_test, close_test]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
merged_test_df.to_csv("./balanced_test_set_correspondence.csv", index=False)

# 2. Split remaining into Train (80%) and Validation (20%)
pos_train, pos_val = train_test_split(pos_rem, test_size=0.2, random_state=RANDOM_STATE)
far_train, far_val = train_test_split(far_rem, test_size=0.2, random_state=RANDOM_STATE)
close_train, close_val = train_test_split(close_rem, test_size=0.2, random_state=RANDOM_STATE)

# 3. Assign User Texts for Train & Val from their respective pools
train_user_pool = pos_train["user_text"].tolist()
val_user_pool = pos_val["user_text"].tolist()

far_train["user_text"] = np.random.choice(train_user_pool, size=len(far_train), replace=True)
close_train["user_text"] = np.random.choice(train_user_pool, size=len(close_train), replace=True)

far_val["user_text"] = np.random.choice(val_user_pool, size=len(far_val), replace=True)
close_val["user_text"] = np.random.choice(val_user_pool, size=len(close_val), replace=True)

# 4. MERGE WITHOUT INFLATION
neg_train_combined = pd.concat([far_train, close_train])
neg_val_combined = pd.concat([far_val, close_val])

merged_train_df = pd.concat([pos_train, neg_train_combined]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
merged_val_df = pd.concat([pos_val, neg_val_combined]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# 5. Calculate the Class Weight for the Neural Network
# Formula: (Count of Majority Class) / (Count of Minority Class)
class_1_weight = len(neg_train_combined) / len(pos_train)

print("--- UN-INFLATED DATASET SIZES ---")
print(f"Train label counts:\n{merged_train_df['label'].value_counts().to_string()}\n")
print(f"Val label counts:\n{merged_val_df['label'].value_counts().to_string()}\n")
print(f"Test label counts:\n{merged_test_df['label'].value_counts().to_string()}\n")
print(f"Calculated Class 1 (Realistic) Weight Multiplier: {class_1_weight:.3f}")

--- UN-INFLATED DATASET SIZES ---
Train label counts:
label
0    1918
1     676

Val label counts:
label
0    480
1    170

Test label counts:
label
1    100
0    100

Calculated Class 1 (Realistic) Weight Multiplier: 2.837


In [ ]:
# Cell 3: Tokenization
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

MODEL_NAME = "BAAI/bge-small-en-v1.5"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_hf = Dataset.from_pandas(merged_train_df[["user_text", "physio_summary", "label"]], preserve_index=False)
val_hf = Dataset.from_pandas(merged_val_df[["user_text", "physio_summary", "label"]], preserve_index=False)

def tokenize_fn(batch):
    return tokenizer(
        batch["user_text"],
        batch["physio_summary"],
        truncation=True,
        max_length=256
    )

train_tokenized = train_hf.map(tokenize_fn, batched=True)
val_tokenized = val_hf.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Map:   0%|          | 0/2594 [00:00<?, ? examples/s]

Map:   0%|          | 0/650 [00:00<?, ? examples/s]

In [ ]:
# Cell 4: Train BGE-Small with Class-Weighted Loss
import torch
import torch.nn as nn
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

SAVE_PATH = "/content/drive/MyDrive/bge_physio_correspondence_class_weighted"

# Load the stable BGE-small model
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1_macro": f1}

# 1. Apply the Class Weights to a PyTorch Tensor
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Class 0 gets standard weight (1.0). Class 1 gets the amplified weight (~2.17).
weights_tensor = torch.tensor([1.0, class_1_weight]).to(device)

# 2. Define the Custom Trainer
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # Extract labels
        labels = inputs.pop("labels")
        # Forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # Apply the mathematically weighted CrossEntropyLoss
        loss_fct = nn.CrossEntropyLoss(weight=weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

# 3. Setup Training Arguments
training_args = TrainingArguments(
    output_dir="./bge_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_steps=45,             # Reduced to match 15% of the new, smaller dataset
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none"
)

# Initialize the Custom WeightedTrainer
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Fine-tuned model successfully saved to: {SAVE_PATH}")

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: BAAI/bge-small-en-v1.5
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.646126,0.509148,0.661538,0.653991
2,0.444515,0.389428,0.836923,0.809660
3,0.368900,0.336957,0.827692,0.808453
4,0.329181,0.322638,0.850769,0.831263
5,0.280937,0.296705,0.864615,0.843245


... (3 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model successfully saved to: /content/drive/MyDrive/bge_physio_correspondence_class_weighted


In [ ]:
# Cell 5: Load fine-tuned BGE-small model from Drive and evaluate
import pandas as pd
import numpy as np
from scipy.special import softmax
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, DataCollatorWithPadding
from datasets import Dataset

SAVE_PATH = "/content/drive/MyDrive/bge_physio_correspondence_class_weighted"
TEST_SET_PATH = "./balanced_test_set_correspondence.csv"
RESULTS_SAVE_PATH = "./correspondence_test_results.csv"

print(f"Loading fine-tuned BGE model and tokenizer from: {SAVE_PATH}")
tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)
model = AutoModelForSequenceClassification.from_pretrained(SAVE_PATH)

test_df = pd.read_csv(TEST_SET_PATH)
test_hf = Dataset.from_pandas(test_df[["user_text", "physio_summary", "label"]], preserve_index=False)

def tokenize_fn(batch):
    return tokenizer(
        batch["user_text"],
        batch["physio_summary"],
        truncation=True,
        max_length=256
    )

test_tokenized = test_hf.map(tokenize_fn, batched=True)

# FIXED: Initialize the dynamic data collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# FIXED: Pass the data collator to the inference Trainer
inference_trainer = Trainer(
    model=model,
    data_collator=data_collator
)

print("Running predictions on the test set...")
predictions = inference_trainer.predict(test_tokenized)

logits = predictions.predictions
probs = softmax(logits, axis=1)

test_df["prob_unrealistic"] = probs[:, 0]
test_df["prob_realistic"] = probs[:, 1]
test_df["predicted_label"] = np.argmax(logits, axis=1)
test_df["flagged_by_BGE"] = test_df["predicted_label"] == 0

print("\n" + "=" * 60)
print("BGE-SMALL TEST SET METRICS (LOADED FROM DRIVE)")
print("=" * 60)

print("\nClassification Report:")
print(classification_report(
    test_df["label"],
    test_df["predicted_label"],
    target_names=["Unrealistic (0)", "Realistic (1)"]
))

print("\nConfusion Matrix:")
cm = confusion_matrix(test_df["label"], test_df["predicted_label"])
print(f"True Negatives (Unrealistic detected):  {cm[0][0]}")
print(f"False Positives (Unrealistic as Real): {cm[0][1]}")
print(f"False Negatives (Realistic as Unreal): {cm[1][0]}")
print(f"True Positives (Realistic detected):   {cm[1][1]}")

test_df.to_csv(RESULTS_SAVE_PATH, index=False)
print(f"\nSaved test predictions to: {RESULTS_SAVE_PATH}")

Loading fine-tuned BGE model and tokenizer from: /content/drive/MyDrive/bge_physio_correspondence_class_weighted


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Running predictions on the test set...



BGE-SMALL TEST SET METRICS (LOADED FROM DRIVE)

Classification Report:
                 precision    recall  f1-score   support

Unrealistic (0)       0.90      0.86      0.88       100
  Realistic (1)       0.87      0.90      0.88       100

       accuracy                           0.88       200
      macro avg       0.88      0.88      0.88       200
   weighted avg       0.88      0.88      0.88       200


Confusion Matrix:
True Negatives (Unrealistic detected):  86
False Positives (Unrealistic as Real): 14
False Negatives (Realistic as Unreal): 10
True Positives (Realistic detected):   90

Saved test predictions to: ./correspondence_test_results.csv


In [ ]:
# Cell 6: Breakdown by Close Variant, Far Variant, and Realistic
from IPython.display import display
import pandas as pd

df_res = pd.read_csv("./correspondence_test_results.csv")
categories = sorted(df_res["category"].unique())

rows = []
for flag in [True, False]:
    row = {
        "Method": "BGE-small-NLI",
        "Flagged as Unrealistic": flag
    }
    for cat in categories:
        # FIXED: Updated condition to check the new BGE column
        mask = (df_res["category"] == cat) & (df_res["flagged_by_BGE"] == flag)
        row[cat] = int(mask.sum())
    rows.append(row)

summary_table = pd.DataFrame(rows)

print("=" * 80)
print("METHOD-WISE RESULTS BY CATEGORY (CLOSE vs FAR vs REALISTIC)")
print("=" * 80)
display(summary_table)
summary_table.to_csv("./correspondence_category_breakdown.csv", index=False)

METHOD-WISE RESULTS BY CATEGORY (CLOSE vs FAR vs REALISTIC)


,Method,Flagged as Unrealistic,Close Variant,Far Variant,Realistic
0,BGE-small-NLI,True,43,43,10
1,BGE-small-NLI,False,7,7,90


In the above method, for **data Inflation** of Realistic training set (pos_train), which only has about 846 pairs, the following steps are followed to make it match the 1,840 Unrealistic pairs [the **resample() function** essentially acts like drawing numbers from a hat]:

1) It randomly picks one of the 846 Realistic pairs and copies it into the new pos_train_inf dataset.

2) Because replace=True is enabled, it puts that pair back into the hat so it can potentially be drawn again.

It repeats this process exactly 1,840 times.

However, if this method of data inflation causes Over-fitting, below code can be used instead.

**The Alternative:** Class-Weighted Loss


To balance the classes without duplicating data, the best machine learning practice is Class Weighting.

Instead of duplicating the 846 Realistic rows to match the 1,840 Unrealistic rows:
1) Leave the dataset un-inflated: Train on the original ~846 Realistic pairs and ~1,840 Unrealistic pairs with no duplicates.
2) Penalize errors proportionally: Modify the loss function so that whenever the model misclassifies a "Realistic" pair, the loss penalty is multiplied by roughly $2.17\times$ ($\frac{1840}{846}$).

# **CHECK 3:** **Using User text, Physio Summary and AI response**

In [ ]:
# Cell 0: Install and import dependencies
!pip install -q transformers datasets accelerate scikit-learn pandas numpy torch

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from scipy.special import softmax
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from IPython.display import display

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

PyTorch Version: 2.11.0+cu128
CUDA Available: True


In [ ]:
# Cell 1: Load datasets, deduplicate, and assign unique external context pairs
REALISTIC_PATH = "/content/drive/MyDrive/user_text_physio_summaries_ai_response.csv"
FAR_PATH = "/content/drive/MyDrive/far_variant_data_clean.csv"
CLOSE_PATH = "/content/drive/MyDrive/close_variant_impossible_point.csv"
EXTERNAL_TEXTS_PATH = "/content/drive/MyDrive/user_texts_ai_resp_frm_Extes.csv"

# 1. Load Realistic Data (Now requiring ai_response)
pos_df = pd.read_csv(REALISTIC_PATH)[["user_text", "ai_text", "physio_summary"]].dropna()
pos_df = pos_df.drop_duplicates(subset=["user_text", "ai_text", "physio_summary"]).reset_index(drop=True)
pos_df["category"] = "Realistic"
pos_df["label"] = 1

# 2. Load External Context Pairs (user_text + ai_response)
ext_df = pd.read_csv(EXTERNAL_TEXTS_PATH)[["user_text", "ai_text"]].dropna()
ext_df = ext_df.drop_duplicates(subset=["user_text", "ai_text"]).reset_index(drop=True)

# 3. Load Negative Summaries
far_df = pd.read_csv(FAR_PATH)[["physio_summary"]].dropna()
far_df = far_df.drop_duplicates(subset=["physio_summary"]).reset_index(drop=True)
far_df["category"] = "Far Variant"
far_df["label"] = 0

close_df = pd.read_csv(CLOSE_PATH)[["physio_summary"]].dropna()
close_df = close_df.drop_duplicates(subset=["physio_summary"]).reset_index(drop=True)
close_df["category"] = "Close Variant"
close_df["label"] = 0

# 4. Sample and Assign Unique External Pairs
total_needed = len(far_df) + len(close_df)
if total_needed > len(ext_df):
    raise ValueError(f"Not enough unique external pairs! Need {total_needed}, have {len(ext_df)}.")

# Sample indices without replacement
sampled_indices = np.random.choice(ext_df.index, size=total_needed, replace=False)
sampled_ext_df = ext_df.iloc[sampled_indices].reset_index(drop=True)

# Split and assign to Far and Close
far_df["user_text"] = sampled_ext_df.iloc[:len(far_df)]["user_text"].values
far_df["ai_text"] = sampled_ext_df.iloc[:len(far_df)]["ai_text"].values

close_df["user_text"] = sampled_ext_df.iloc[len(far_df):]["user_text"].values
close_df["ai_text"] = sampled_ext_df.iloc[len(far_df):]["ai_text"].values

# Reorder columns
far_df = far_df[["user_text", "ai_text", "physio_summary", "category", "label"]]
close_df = close_df[["user_text", "ai_text", "physio_summary", "category", "label"]]

print(f"Loaded Clean Summaries & Assigned Context:")
print(f"- Realistic Pairs: {len(pos_df)}")
print(f"- Far Variants (Mismatched): {len(far_df)}")
print(f"- Close Variants (Mismatched): {len(close_df)}")

Loaded Clean Summaries & Assigned Context:
- Realistic Pairs: 946
- Far Variants (Mismatched): 1500
- Close Variants (Mismatched): 998


In [ ]:
# Cell 2: Test split and class weight calculation (NO INFLATION)
# 1. Held-out Test Set (100 Realistic, 50 Far, 50 Close)
pos_test = pos_df.sample(n=100, random_state=RANDOM_STATE)
pos_rem = pos_df.drop(pos_test.index).reset_index(drop=True)

far_test = far_df.sample(n=50, random_state=RANDOM_STATE)
far_rem = far_df.drop(far_test.index).reset_index(drop=True)

close_test = close_df.sample(n=50, random_state=RANDOM_STATE)
close_rem = close_df.drop(close_test.index).reset_index(drop=True)

merged_test_df = pd.concat([pos_test, far_test, close_test]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
merged_test_df.to_csv("./balanced_test_set_correspondence.csv", index=False)

# 2. Split remaining into Train (80%) and Validation (20%)
pos_train, pos_val = train_test_split(pos_rem, test_size=0.2, random_state=RANDOM_STATE)
far_train, far_val = train_test_split(far_rem, test_size=0.2, random_state=RANDOM_STATE)
close_train, close_val = train_test_split(close_rem, test_size=0.2, random_state=RANDOM_STATE)

# 3. MERGE WITHOUT INFLATION
neg_train_combined = pd.concat([far_train, close_train])
neg_val_combined = pd.concat([far_val, close_val])

merged_train_df = pd.concat([pos_train, neg_train_combined]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
merged_val_df = pd.concat([pos_val, neg_val_combined]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# 4. Calculate the Class Weight for the Neural Network
class_1_weight = len(neg_train_combined) / len(pos_train)

print("--- UN-INFLATED DATASET SIZES ---")
print(f"Train label counts:\n{merged_train_df['label'].value_counts().to_string()}\n")
print(f"Val label counts:\n{merged_val_df['label'].value_counts().to_string()}\n")
print(f"Calculated Class 1 (Realistic) Weight Multiplier: {class_1_weight:.3f}")

--- UN-INFLATED DATASET SIZES ---
Train label counts:
label
0    1918
1     676

Val label counts:
label
0    480
1    170

Calculated Class 1 (Realistic) Weight Multiplier: 2.837


In [ ]:
# Cell 3: Tokenization with 3-part sequence construction
MODEL_NAME = "BAAI/bge-small-en-v1.5"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_hf = Dataset.from_pandas(merged_train_df[["user_text", "ai_text", "physio_summary", "label"]], preserve_index=False)
val_hf = Dataset.from_pandas(merged_val_df[["user_text", "ai_text", "physio_summary", "label"]], preserve_index=False)

def tokenize_fn(batch):
    # Combine User Text and AI Response with a SEP token
    combined_context = [
        f"{u} {tokenizer.sep_token} {a}"
        for u, a in zip(batch["user_text"], batch["ai_text"])
    ]
    # Pass combined context as Text A, and Physio Summary as Text B
    return tokenizer(
        combined_context,
        batch["physio_summary"],
        truncation=True,
        max_length=384 # Increased slightly to accommodate AI response length
    )

train_tokenized = train_hf.map(tokenize_fn, batched=True)
val_tokenized = val_hf.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Map:   0%|          | 0/2594 [00:00<?, ? examples/s]

Map:   0%|          | 0/650 [00:00<?, ? examples/s]

In [ ]:
# Cell 4: Train BGE-Small with Class-Weighted Loss
SAVE_PATH = "/content/drive/MyDrive/bge_physio_correspondence_also_including_ai_response"

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1_macro": f1}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights_tensor = torch.tensor([1.0, class_1_weight]).to(device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir="./bge_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_steps=45,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Fine-tuned model successfully saved to: {SAVE_PATH}")

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: BAAI/bge-small-en-v1.5
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.403227,0.151777,0.978462,0.971684
2,0.092007,0.095596,0.981538,0.976103
3,0.036260,0.316514,0.956923,0.940958
4,0.024820,0.117621,0.984615,0.979774
5,0.016133,0.122393,0.984615,0.979774


... (3 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model successfully saved to: /content/drive/MyDrive/bge_physio_correspondence_also_including_ai_response


In [ ]:
# Cell 5: Load fine-tuned BGE-small model from Drive and evaluate
TEST_SET_PATH = "./balanced_test_set_correspondence.csv"
RESULTS_SAVE_PATH = "./correspondence_test_results.csv"

print(f"Loading fine-tuned BGE model and tokenizer from: {SAVE_PATH}")
tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)
model = AutoModelForSequenceClassification.from_pretrained(SAVE_PATH)

test_df = pd.read_csv(TEST_SET_PATH)
test_hf = Dataset.from_pandas(test_df[["user_text", "ai_text", "physio_summary", "label"]], preserve_index=False)

def tokenize_fn_test(batch):
    combined_context = [
        f"{u} {tokenizer.sep_token} {a}"
        for u, a in zip(batch["user_text"], batch["ai_text"])
    ]
    return tokenizer(
        combined_context,
        batch["physio_summary"],
        truncation=True,
        max_length=384
    )

test_tokenized = test_hf.map(tokenize_fn_test, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

inference_trainer = Trainer(
    model=model,
    data_collator=data_collator
)

print("Running predictions on the test set...")
predictions = inference_trainer.predict(test_tokenized)
logits = predictions.predictions
probs = softmax(logits, axis=1)

test_df["prob_unrealistic"] = probs[:, 0]
test_df["prob_realistic"] = probs[:, 1]
test_df["predicted_label"] = np.argmax(logits, axis=1)
test_df["flagged_by_BGE"] = test_df["predicted_label"] == 0

print("\n" + "=" * 60)
print("BGE-SMALL TEST SET METRICS (LOADED FROM DRIVE)")
print("=" * 60)

print("\nClassification Report:")
print(classification_report(
    test_df["label"],
    test_df["predicted_label"],
    target_names=["Unrealistic (0)", "Realistic (1)"]
))

print("\nConfusion Matrix:")
cm = confusion_matrix(test_df["label"], test_df["predicted_label"])
print(f"True Negatives (Unrealistic detected):  {cm[0][0]}")
print(f"False Positives (Unrealistic as Real): {cm[0][1]}")
print(f"False Negatives (Realistic as Unreal): {cm[1][0]}")
print(f"True Positives (Realistic detected):   {cm[1][1]}")

test_df.to_csv(RESULTS_SAVE_PATH, index=False)

Loading fine-tuned BGE model and tokenizer from: /content/drive/MyDrive/bge_physio_correspondence_also_including_ai_response


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Running predictions on the test set...



BGE-SMALL TEST SET METRICS (LOADED FROM DRIVE)

Classification Report:
                 precision    recall  f1-score   support

Unrealistic (0)       0.96      1.00      0.98       100
  Realistic (1)       1.00      0.96      0.98       100

       accuracy                           0.98       200
      macro avg       0.98      0.98      0.98       200
   weighted avg       0.98      0.98      0.98       200


Confusion Matrix:
True Negatives (Unrealistic detected):  100
False Positives (Unrealistic as Real): 0
False Negatives (Realistic as Unreal): 4
True Positives (Realistic detected):   96


In [ ]:
# Cell 6: Breakdown by Close Variant, Far Variant, and Realistic
df_res = pd.read_csv("./correspondence_test_results.csv")
categories = sorted(df_res["category"].unique())

rows = []
for flag in [True, False]:
    row = {
        "Method": "BGE-small-NLI",
        "Flagged as Unrealistic": flag
    }
    for cat in categories:
        mask = (df_res["category"] == cat) & (df_res["flagged_by_BGE"] == flag)
        row[cat] = int(mask.sum())
    rows.append(row)

summary_table = pd.DataFrame(rows)

print("=" * 80)
print("METHOD-WISE RESULTS BY CATEGORY (CLOSE vs FAR vs REALISTIC)")
print("=" * 80)
display(summary_table)
summary_table.to_csv("./correspondence_category_breakdown.csv", index=False)

METHOD-WISE RESULTS BY CATEGORY (CLOSE vs FAR vs REALISTIC)


,Method,Flagged as Unrealistic,Close Variant,Far Variant,Realistic
0,BGE-small-NLI,True,50,50,4
1,BGE-small-NLI,False,0,0,96


# **Using User text, Physio Summary and improved AI response**

In [ ]:
# Cell 0: Install and import dependencies
!pip install -q transformers datasets accelerate scikit-learn pandas numpy torch

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from scipy.special import softmax
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from IPython.display import display

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

PyTorch Version: 2.11.0+cu128
CUDA Available: True


In [ ]:
# Cell 1: Load datasets, deduplicate, and assign unique external context pairs
REALISTIC_PATH = "/content/drive/MyDrive/user_text_physio_summaries_improved_ai_response.csv"
FAR_PATH = "/content/drive/MyDrive/far_variant_data_clean.csv"
CLOSE_PATH = "/content/drive/MyDrive/close_variant_impossible_point.csv"
EXTERNAL_TEXTS_PATH = "/content/drive/MyDrive/user_texts_ai_resp_frm_Extes.csv"

# 1. Load Realistic Data (Now requiring ai_response)
pos_df = pd.read_csv(REALISTIC_PATH)[["user_text", "ai_text", "physio_summary"]].dropna()
pos_df = pos_df.drop_duplicates(subset=["user_text", "ai_text", "physio_summary"]).reset_index(drop=True)
pos_df["category"] = "Realistic"
pos_df["label"] = 1

# 2. Load External Context Pairs (user_text + ai_response)
ext_df = pd.read_csv(EXTERNAL_TEXTS_PATH)[["user_text", "ai_text"]].dropna()
ext_df = ext_df.drop_duplicates(subset=["user_text", "ai_text"]).reset_index(drop=True)

# 3. Load Negative Summaries
far_df = pd.read_csv(FAR_PATH)[["physio_summary"]].dropna()
far_df = far_df.drop_duplicates(subset=["physio_summary"]).reset_index(drop=True)
far_df["category"] = "Far Variant"
far_df["label"] = 0

close_df = pd.read_csv(CLOSE_PATH)[["physio_summary"]].dropna()
close_df = close_df.drop_duplicates(subset=["physio_summary"]).reset_index(drop=True)
close_df["category"] = "Close Variant"
close_df["label"] = 0

# 4. Sample and Assign Unique External Pairs
total_needed = len(far_df) + len(close_df)
if total_needed > len(ext_df):
    raise ValueError(f"Not enough unique external pairs! Need {total_needed}, have {len(ext_df)}.")

# Sample indices without replacement
sampled_indices = np.random.choice(ext_df.index, size=total_needed, replace=False)
sampled_ext_df = ext_df.iloc[sampled_indices].reset_index(drop=True)

# Split and assign to Far and Close
far_df["user_text"] = sampled_ext_df.iloc[:len(far_df)]["user_text"].values
far_df["ai_text"] = sampled_ext_df.iloc[:len(far_df)]["ai_text"].values

close_df["user_text"] = sampled_ext_df.iloc[len(far_df):]["user_text"].values
close_df["ai_text"] = sampled_ext_df.iloc[len(far_df):]["ai_text"].values

# Reorder columns
far_df = far_df[["user_text", "ai_text", "physio_summary", "category", "label"]]
close_df = close_df[["user_text", "ai_text", "physio_summary", "category", "label"]]

print(f"Loaded Clean Summaries & Assigned Context:")
print(f"- Realistic Pairs: {len(pos_df)}")
print(f"- Far Variants (Mismatched): {len(far_df)}")
print(f"- Close Variants (Mismatched): {len(close_df)}")

Loaded Clean Summaries & Assigned Context:
- Realistic Pairs: 536
- Far Variants (Mismatched): 1500
- Close Variants (Mismatched): 998


In [ ]:
# Cell 2: Test split and class weight calculation (NO INFLATION)
# 1. Held-out Test Set (100 Realistic, 50 Far, 50 Close)
pos_test = pos_df.sample(n=100, random_state=RANDOM_STATE)
pos_rem = pos_df.drop(pos_test.index).reset_index(drop=True)

far_test = far_df.sample(n=50, random_state=RANDOM_STATE)
far_rem = far_df.drop(far_test.index).reset_index(drop=True)

close_test = close_df.sample(n=50, random_state=RANDOM_STATE)
close_rem = close_df.drop(close_test.index).reset_index(drop=True)

merged_test_df = pd.concat([pos_test, far_test, close_test]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
merged_test_df.to_csv("./balanced_test_set_correspondence.csv", index=False)

# 2. Split remaining into Train (80%) and Validation (20%)
pos_train, pos_val = train_test_split(pos_rem, test_size=0.2, random_state=RANDOM_STATE)
far_train, far_val = train_test_split(far_rem, test_size=0.2, random_state=RANDOM_STATE)
close_train, close_val = train_test_split(close_rem, test_size=0.2, random_state=RANDOM_STATE)

# 3. MERGE WITHOUT INFLATION
neg_train_combined = pd.concat([far_train, close_train])
neg_val_combined = pd.concat([far_val, close_val])

merged_train_df = pd.concat([pos_train, neg_train_combined]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
merged_val_df = pd.concat([pos_val, neg_val_combined]).sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

# 4. Calculate the Class Weight for the Neural Network
class_1_weight = len(neg_train_combined) / len(pos_train)

print("--- UN-INFLATED DATASET SIZES ---")
print(f"Train label counts:\n{merged_train_df['label'].value_counts().to_string()}\n")
print(f"Val label counts:\n{merged_val_df['label'].value_counts().to_string()}\n")
print(f"Calculated Class 1 (Realistic) Weight Multiplier: {class_1_weight:.3f}")

--- UN-INFLATED DATASET SIZES ---
Train label counts:
label
0    1918
1     348

Val label counts:
label
0    480
1     88

Calculated Class 1 (Realistic) Weight Multiplier: 5.511


In [ ]:
# Cell 3: Tokenization with 3-part sequence construction
MODEL_NAME = "BAAI/bge-small-en-v1.5"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_hf = Dataset.from_pandas(merged_train_df[["user_text", "ai_text", "physio_summary", "label"]], preserve_index=False)
val_hf = Dataset.from_pandas(merged_val_df[["user_text", "ai_text", "physio_summary", "label"]], preserve_index=False)

def tokenize_fn(batch):
    # Combine User Text and AI Response with a SEP token
    combined_context = [
        f"{u} {tokenizer.sep_token} {a}"
        for u, a in zip(batch["user_text"], batch["ai_text"])
    ]
    # Pass combined context as Text A, and Physio Summary as Text B
    return tokenizer(
        combined_context,
        batch["physio_summary"],
        truncation=True,
        max_length=384 # Increased slightly to accommodate AI response length
    )

train_tokenized = train_hf.map(tokenize_fn, batched=True)
val_tokenized = val_hf.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Map:   0%|          | 0/2266 [00:00<?, ? examples/s]

Map:   0%|          | 0/568 [00:00<?, ? examples/s]

In [ ]:
# Cell 4: Train BGE-Small with Class-Weighted Loss
SAVE_PATH = "/content/drive/MyDrive/bge_physio_correspondence_also_including_improved_ai_response"

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1_macro": f1}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights_tensor = torch.tensor([1.0, class_1_weight]).to(device)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = nn.CrossEntropyLoss(weight=weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir="./bge_checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    warmup_steps=45,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"Fine-tuned model successfully saved to: {SAVE_PATH}")

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1`.


model.safetensors: reconstructing file:   0%|          |  0.00B /  133MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: BAAI/bge-small-en-v1.5
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.375741,0.112184,0.991197,0.982789
2,0.029250,0.221320,0.982394,0.964725
3,0.018035,0.080232,0.992958,0.986297
4,0.006673,0.086344,0.992958,0.986297
5,0.005436,0.080411,0.994718,0.989772


... (3 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model successfully saved to: /content/drive/MyDrive/bge_physio_correspondence_also_including_improved_ai_response


In [ ]:
# Cell 5: Load fine-tuned BGE-small model from Drive and evaluate
TEST_SET_PATH = "./balanced_test_set_correspondence.csv"
RESULTS_SAVE_PATH = "./correspondence_test_results.csv"

print(f"Loading fine-tuned BGE model and tokenizer from: {SAVE_PATH}")
tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)
model = AutoModelForSequenceClassification.from_pretrained(SAVE_PATH)

test_df = pd.read_csv(TEST_SET_PATH)
test_hf = Dataset.from_pandas(test_df[["user_text", "ai_text", "physio_summary", "label"]], preserve_index=False)

def tokenize_fn_test(batch):
    combined_context = [
        f"{u} {tokenizer.sep_token} {a}"
        for u, a in zip(batch["user_text"], batch["ai_text"])
    ]
    return tokenizer(
        combined_context,
        batch["physio_summary"],
        truncation=True,
        max_length=384
    )

test_tokenized = test_hf.map(tokenize_fn_test, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

inference_trainer = Trainer(
    model=model,
    data_collator=data_collator
)

print("Running predictions on the test set...")
predictions = inference_trainer.predict(test_tokenized)
logits = predictions.predictions
probs = softmax(logits, axis=1)

test_df["prob_unrealistic"] = probs[:, 0]
test_df["prob_realistic"] = probs[:, 1]
test_df["predicted_label"] = np.argmax(logits, axis=1)
test_df["flagged_by_BGE"] = test_df["predicted_label"] == 0

print("\n" + "=" * 60)
print("BGE-SMALL TEST SET METRICS (LOADED FROM DRIVE)")
print("=" * 60)

print("\nClassification Report:")
print(classification_report(
    test_df["label"],
    test_df["predicted_label"],
    target_names=["Unrealistic (0)", "Realistic (1)"]
))

print("\nConfusion Matrix:")
cm = confusion_matrix(test_df["label"], test_df["predicted_label"])
print(f"True Negatives (Unrealistic detected):  {cm[0][0]}")
print(f"False Positives (Unrealistic as Real): {cm[0][1]}")
print(f"False Negatives (Realistic as Unreal): {cm[1][0]}")
print(f"True Positives (Realistic detected):   {cm[1][1]}")

test_df.to_csv(RESULTS_SAVE_PATH, index=False)

Loading fine-tuned BGE model and tokenizer from: /content/drive/MyDrive/bge_physio_correspondence_also_including_improved_ai_response


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Running predictions on the test set...



BGE-SMALL TEST SET METRICS (LOADED FROM DRIVE)

Classification Report:
                 precision    recall  f1-score   support

Unrealistic (0)       0.98      1.00      0.99       100
  Realistic (1)       1.00      0.98      0.99       100

       accuracy                           0.99       200
      macro avg       0.99      0.99      0.99       200
   weighted avg       0.99      0.99      0.99       200


Confusion Matrix:
True Negatives (Unrealistic detected):  100
False Positives (Unrealistic as Real): 0
False Negatives (Realistic as Unreal): 2
True Positives (Realistic detected):   98


In [ ]:
# Cell 6: Breakdown by Close Variant, Far Variant, and Realistic
df_res = pd.read_csv("./correspondence_test_results.csv")
categories = sorted(df_res["category"].unique())

rows = []
for flag in [True, False]:
    row = {
        "Method": "BGE-small-NLI",
        "Flagged as Unrealistic": flag
    }
    for cat in categories:
        mask = (df_res["category"] == cat) & (df_res["flagged_by_BGE"] == flag)
        row[cat] = int(mask.sum())
    rows.append(row)

summary_table = pd.DataFrame(rows)

print("=" * 80)
print("METHOD-WISE RESULTS BY CATEGORY (CLOSE vs FAR vs REALISTIC)")
print("=" * 80)
display(summary_table)
summary_table.to_csv("./correspondence_category_breakdown.csv", index=False)

METHOD-WISE RESULTS BY CATEGORY (CLOSE vs FAR vs REALISTIC)


,Method,Flagged as Unrealistic,Close Variant,Far Variant,Realistic
0,BGE-small-NLI,True,50,50,2
1,BGE-small-NLI,False,0,0,98
